# PTM literature and source landscape

For **human phospho** rows in `ptm.txt` that overlap measured sites, summarizes **PMID multiplicity** and **source** × **ptm_type** counts.

**Per condition (below):** the same PTM rows are filtered to sites that are **significant in that stimulation** (nominal p &lt; 0.05, |Log2FC| &gt; 0) so you can see where literature-cited evidence concentrates across arms.

**Inputs:** `data.tsv`, `data/ptm.txt`.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    from IPython.display import display
except ImportError:
    display = print


def find_project_root() -> Path:
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / "data.tsv").is_file() and (p / "analyze" / "_conditions.py").is_file():
            return p
    raise FileNotFoundError("Project root not found (need data.tsv + analyze/_conditions.py).")


ROOT = find_project_root()
sys.path.insert(0, str(ROOT / "analyze"))
from _conditions import CONDITIONS  # noqa: E402
import ptm_data as ptm  # noqa: E402

DATA_PATH = ROOT / "data.tsv"
PTM_PATH = ROOT / "data" / "ptm.txt"
PPIC_PATH = ROOT / "data" / "ppic" / "edges.tsv"
KSEA_LONG = ROOT / "output" / "analyze" / "ksea_prerank_long.tsv"

THR = float(-np.log10(0.05))
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 40)

print("ROOT =", ROOT)


In [ ]:
DATA = pd.read_csv(DATA_PATH, sep='\t', low_memory=False)
df_ptm = ptm.load_ptm(PTM_PATH)
df_phos = ptm.add_site_key_ptm(ptm.filter_human(ptm.filter_ptm_type(df_ptm, 'PHOSPHORYLATION')))
data_sk = ptm.add_site_key_phospho(DATA)
keys = set(data_sk['_site_key']) - {''}
ov = df_phos[df_phos['_site_key'].isin(keys)].copy()
ov['_n_pmids'] = ov['pmid'].apply(ptm.n_pmids)
print('Overlapping phospho PTM rows:', len(ov))


## Per condition (PTM rows linked to *significant* sites only)

For each `CONDITIONS` arm: keep overlapping `ptm.txt` rows whose **site** is significant in *that* condition (same rule as other notebooks: −log10 p ≥ −log10 0.05, |Log2FC| &gt; 0). The **bar chart** sums `_n_pmids` over those rows; the **heatmap** counts PTM **rows** by `source` and condition.

In [ ]:
per_rows: list[dict] = []
for c in CONDITIONS:
    log2 = pd.to_numeric(data_sk[c.log2fc_col], errors="coerce")
    nlp = pd.to_numeric(data_sk[c.neglogp_col], errors="coerce")
    sig = (nlp >= THR) & log2.notna() & (np.abs(log2) > 0)
    sig_keys = set(data_sk.loc[sig, "_site_key"])
    ovc = ov[ov["_site_key"].isin(sig_keys)]
    per_rows.append(
        {
            "condition": c.condition_id,
            "n_sig_sites": int(sig.sum()),
            "n_ptm_rows": len(ovc),
            "sum_distinct_pmid_units": int(ovc["_n_pmids"].sum()) if len(ovc) else 0,
            "mean_n_pmids_per_row": float(ovc["_n_pmids"].mean()) if len(ovc) else float("nan"),
        }
    )
per_df = pd.DataFrame(per_rows)
display(per_df)

fig, ax = plt.subplots(figsize=(10, 3.8))
sns.barplot(data=per_df, x="condition", y="sum_distinct_pmid_units", ax=ax, color="seagreen")
ax.set_ylabel("Sum of PMID-count units (overlapping PTM rows for sig sites)")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()

# Source x condition: row counts (same sig-site filter per column)
sc_parts: list[pd.DataFrame] = []
for c in CONDITIONS:
    log2 = pd.to_numeric(data_sk[c.log2fc_col], errors="coerce")
    nlp = pd.to_numeric(data_sk[c.neglogp_col], errors="coerce")
    sig = (nlp >= THR) & log2.notna() & (np.abs(log2) > 0)
    sig_keys = set(data_sk.loc[sig, "_site_key"])
    ovc = ov[ov["_site_key"].isin(sig_keys)]
    if ovc.empty:
        continue
    vc = ovc["source"].astype(str).value_counts()
    sc_parts.append(pd.DataFrame({c.condition_id: vc}))
if sc_parts:
    pivot = pd.concat(sc_parts, axis=1).fillna(0).astype(int)
    fig, ax = plt.subplots(figsize=(12, max(4, 0.22 * len(pivot))))
    sns.heatmap(pivot, ax=ax, cmap="YlGnBu", linewidths=0.2)
    ax.set_title("PTM database source × condition (row counts; sig sites only)")
    fig.tight_layout()
    plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ov['_n_pmids'].clip(upper=20).hist(bins=21, ax=ax, color='slategray')
ax.set_xlabel('Distinct PMIDs per row (capped at 20 for display)')
ax.set_ylabel('Rows')
fig.tight_layout()
plt.show()


In [ ]:
by_gene = (
    ov.assign(sub=ov['substrate_genename'].astype(str).str.upper())
    .groupby('sub')['_n_pmids']
    .sum()
    .sort_values(ascending=False)
    .head(20)
)
fig, ax = plt.subplots(figsize=(7, 4))
by_gene.sort_values().plot(kind='barh', ax=ax, color='darkgreen')
ax.set_xlabel('Sum of distinct-PMID counts (overlapping rows)')
fig.tight_layout()
plt.show()


In [ ]:
ct = pd.crosstab(ov['source'].astype(str), ov['ptm_type'].astype(str))
fig, ax = plt.subplots(figsize=(10, max(4, 0.25 * len(ct))))
sns.heatmap(ct, ax=ax, cmap='magma', linewidths=0.3)
ax.set_title('Source × PTM type (human rows overlapping measured sites)')
fig.tight_layout()
plt.show()
